# KLIG Comprehensive Evaluation — All 9 Methods

In [ ]:
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1.git /content/KLIG_V1 2>/dev/null || \
    (cd /content/KLIG_V1 && git pull) 2>/dev/null
!pip install -e /content/KLIG_V1/infocube-main -q
!pip install datasets tqdm scipy nltk -q
import os, sys
for _root in ['/content/KLIG_V1/infocube-main', 'infocube-main', '.']:
    if os.path.isdir(_root) and _root not in sys.path:
        sys.path.insert(0, _root)


In [ ]:
import importlib, os, sys, math, pickle, warnings, itertools
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import nltk; nltk.download('wordnet', quiet=True)
from nltk.corpus import wordnet as wn
from scipy.stats import spearmanr
from torchvision.models import resnet50, ResNet50_Weights
import torchvision.transforms as T
warnings.filterwarnings('ignore')

# force-reload klig modules
import klig.core.rep_descent_path as _rdp; importlib.reload(_rdp)
import klig.core.ig2_integrator as _ig2; importlib.reload(_ig2)
import klig as _klig; importlib.reload(_klig)

from klig import (KLIntegratedGradients, AttributionResult,
                  GreedyMuAttributor, GreedyJointAttributor,
                  SortedDimPath, DDiffusionPath,
                  RepDescentPath, make_phi_from_layer,
                  KLIGSquared, KLIGSquaredResult)
from klig.core.path import LinearPath
from klig.image.stopping import find_sigma_stop
print('imports OK')


In [ ]:
# ── Dataset / compute budget
N_IMGS     = 100
N_STEPS    = 50
N_SAMPLES  = 10
N_INSERTION_STEPS = 50
N_CURVE_IMGS = 20   # images for path change curves

# ── Sigma
SIGMA_FINAL = 0.25             # σ for KLIG-Linear / DDPath / KL-IG² (fixed)
ADAPTIVE_SIGMA_TAU = 0.95       # confidence threshold for find_sigma_stop

# ── SortedDimPath
GAMMA_LO = 0.25
GAMMA_HI = 4.0
SORTED_DIM_SAMPLES = 32

# ── KL-IG² (RepDescentPath)
T_DESCENT    = 50
LR_MU        = 0.05
LR_LV        = 0.10
N_MC_DESCENT = 16
LOSS_STOP    = 1e-3
LV_FLOOR     = 2 * math.log(1 / 256)
LV_CEIL      = 4.0
MU_MIN, MU_MAX = -2.64, 2.64

# ── Class Sensitivity
CS_PROB_THRESH = 0.05

# ── Visualization
VIS_IMG_IDX = 0
FORCE_RECOMPUTE = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Google Drive cache
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    CACHE_DIR = Path('/content/drive/MyDrive/klig_comprehensive_cache')
except Exception:
    CACHE_DIR = Path('klig_comprehensive_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ['KLIG-Adaptive','KLIG-Linear','DDPath-cos','DDPath-lin','DDPath-quad',
           'Greedy-μ','Greedy-Jt','Greedy-Srt','KL-IG²','KL-IG² (adaptive)']
COLORS = {
    'KLIG-Adaptive': '#2d6a2d', 'KLIG-Linear': '#555555',
    'DDPath-cos': '#1f77b4',    'DDPath-lin':  '#aec7e8',
    'DDPath-quad': '#ff6b6b',   'Greedy-μ':    '#ff7f0e',
    'Greedy-Jt':  '#e377c2',   'Greedy-Srt': '#9467bd',
    'KL-IG²':     '#e41a1c',
    'KL-IG² (adaptive)': '#8b0000',
}
print(f'device: {DEVICE}  |  cache: {CACHE_DIR}')


In [ ]:
weights   = ResNet50_Weights.IMAGENET1K_V2
model     = resnet50(weights=weights).to(DEVICE).eval()
preprocess = weights.transforms()
imagenet_labels = weights.meta['categories']

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
def denormalize(x): return (x.cpu() * _STD + _MEAN).clamp(0,1)

def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0, keepdim=True)
    return a.gather(0, idx).squeeze(0)

def get_sigma_adaptive(x, target):
    return float(min(max(find_sigma_stop(model, x, target=target, tau=ADAPTIVE_SIGMA_TAU), 1/256), 1.0))

# φ for KL-IG²: output of ResNet50 layer4
phi = make_phi_from_layer(model, 'layer4')
print('model loaded  |  phi=layer4')


In [ ]:
_cache_ds = CACHE_DIR / 'dataset.pkl'
if not FORCE_RECOMPUTE and _cache_ds.exists():
    with open(_cache_ds, 'rb') as f: dataset, sigma_per_idx = pickle.load(f)
    print(f'[cache] dataset n={len(dataset)}')
else:
    from datasets import load_dataset as _hf
    _ds = _hf('evanarlian/imagenet_1k_resized_256', split='train', streaming=True)
    dataset = []
    for item in tqdm(_ds.take(N_IMGS * 4), desc='loading'):
        img = item['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)
            tgt  = int(logits.argmax(-1).item())
            conf = logits.softmax(-1)[0, tgt].item()
        if conf > 0.3:
            dataset.append({'x': x, 'target': tgt, 'idx': len(dataset)})
        if len(dataset) >= N_IMGS: break
    # precompute adaptive sigma for each image
    sigma_per_idx = {}
    for row in tqdm(dataset, desc='adaptive sigma'):
        sigma_per_idx[row['idx']] = get_sigma_adaptive(row['x'], row['target'])
    with open(_cache_ds, 'wb') as f: pickle.dump((dataset, sigma_per_idx), f)
    print(f'Collected {len(dataset)} images')
print(f'sigma range: [{min(sigma_per_idx.values()):.4f}, {max(sigma_per_idx.values()):.4f}]')


In [ ]:
# ── Counterfactual selection: target second-most-likely class (for KL-IG²) ───
_cache_cf = CACHE_DIR / 'cf_y2_pool.pkl'

# (1) y_2 for every explicand
y2_per_idx = []
with torch.no_grad():
    for row in dataset:
        x = row['x'].to(DEVICE)
        probs = model(x).softmax(-1)[0]
        top2  = probs.topk(2).indices.tolist()
        y2    = top2[1] if top2[0] == row['target'] else top2[0]
        y2_per_idx.append(int(y2))

needed_classes = set(y2_per_idx)
print(f'distinct y_2 classes needed: {len(needed_classes)}')

# (2) Build a pool with one cf image per needed class
CF_POOL_MAX_SCAN = 8000
if not FORCE_RECOMPUTE and _cache_cf.exists():
    with open(_cache_cf, 'rb') as f: cf_pool_cpu = pickle.load(f)
    print(f'[cache] cf_pool loaded ({len(cf_pool_cpu)} classes)')
else:
    from datasets import load_dataset as _hf
    _ds = _hf('evanarlian/imagenet_1k_resized_256', split='train', streaming=True)
    cf_pool_cpu = {}
    scanned = 0
    pbar = tqdm(total=len(needed_classes), desc='cf pool')
    for item in _ds:
        scanned += 1
        if len(cf_pool_cpu) >= len(needed_classes) or scanned >= CF_POOL_MAX_SCAN:
            break
        img = item['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred = int(model(x).argmax(-1).item())
        if pred in needed_classes and pred not in cf_pool_cpu:
            cf_pool_cpu[pred] = x.cpu()
            pbar.update(1)
    pbar.close()
    with open(_cache_cf, 'wb') as f: pickle.dump(cf_pool_cpu, f)
    print(f'cf_pool built: {len(cf_pool_cpu)}/{len(needed_classes)} classes  '
          f'(scanned {scanned} candidates)')

cf_pool = {k: v.to(DEVICE) for k, v in cf_pool_cpu.items()}

_missing_classes = needed_classes - set(cf_pool.keys())
if _missing_classes:
    print(f'WARNING: {len(_missing_classes)} y_2 classes had no cf image found '
          f'(will use fallback)')

def pick_cf_image(idx):
    y2 = y2_per_idx[idx]
    if y2 in cf_pool:
        return cf_pool[y2]
    own_tgt = dataset[idx]['target']
    for k, v in cf_pool.items():
        if k != own_tgt: return v
    raise RuntimeError('cf_pool empty')

for i in [0, 1, 2]:
    pred_lbl = imagenet_labels[dataset[i]['target']]
    y2_lbl   = imagenet_labels[y2_per_idx[i]]
    matched  = 'ok' if y2_per_idx[i] in cf_pool else '(fallback)'
    print(f'img {i}: predicted={pred_lbl!r:30s}  y_2={y2_lbl!r:30s}  {matched}')


In [ ]:
# ── Attribution loop — all 9 methods ─────────────────────────────────────────
_cache_attr = CACHE_DIR / 'all_attrs.pkl'

if not FORCE_RECOMPUTE and _cache_attr.exists():
    with open(_cache_attr, 'rb') as f: all_attrs = pickle.load(f)
    print('[cache] attrs loaded')
else:
    all_attrs = {m: {} for m in METHODS}

    for row in tqdm(dataset, desc='attributing'):
        x, tgt = row['x'], row['target']
        idx = row['idx']
        x1  = x.squeeze(0).to(DEVICE)
        sig_adapt = sigma_per_idx[idx]

        # KLIG-Adaptive: LinearPath + adaptive sigma
        r = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sig_adapt, path=LinearPath(), device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['KLIG-Adaptive'][idx] = absmax_collapse(r.attr).cpu()

        # KLIG-Linear: LinearPath + fixed sigma
        r = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=SIGMA_FINAL, path=LinearPath(), device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['KLIG-Linear'][idx] = absmax_collapse(r.attr).cpu()

        # DDPath-cos
        r = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=SIGMA_FINAL, path=DDiffusionPath(schedule='cosine'), device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['DDPath-cos'][idx] = absmax_collapse(r.attr).cpu()

        # DDPath-lin
        r = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=SIGMA_FINAL, path=DDiffusionPath(schedule='linear'), device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['DDPath-lin'][idx] = absmax_collapse(r.attr).cpu()

        # DDPath-quad
        r = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=SIGMA_FINAL, path=DDiffusionPath(schedule='quadratic'), device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['DDPath-quad'][idx] = absmax_collapse(r.attr).cpu()

        # Greedy-μ
        r = GreedyMuAttributor(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sig_adapt, device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['Greedy-μ'][idx] = absmax_collapse(r.attr).cpu()

        # Greedy-Jt
        r = GreedyJointAttributor(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sig_adapt, device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['Greedy-Jt'][idx] = absmax_collapse(r.attr).cpu()

        # Greedy-Srt: SortedDimPath + KLIntegratedGradients
        sp = SortedDimPath.from_model_and_input(
            model, x1, target=tgt,
            n_samples=SORTED_DIM_SAMPLES, gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI)
        r = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sig_adapt, path=sp, device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['Greedy-Srt'][idx] = absmax_collapse(r.attr).cpu()

        # KL-IG²: KLIGSquared (forward IG² integration, model-derived baseline)
        x_cf = pick_cf_image(idx)
        if x_cf.dim() == 4: x_cf = x_cf.squeeze(0)
        x_cf = x_cf.to(DEVICE)
        r = KLIGSquared(
            model, phi, x_cf,
            T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
            n_mc_path=N_MC_DESCENT, n_mc_grad=N_SAMPLES,
            sigma_start=SIGMA_FINAL, loss_stop=LOSS_STOP,
            lv_floor=LV_FLOOR, lv_ceil=LV_CEIL,
            mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['KL-IG²'][idx] = absmax_collapse(r.attr).cpu()

        # KL-IG² (adaptive): KLIGSquared with adaptive sigma_start
        lv_floor_adapt = 2 * math.log(sig_adapt)
        r = KLIGSquared(
            model, phi, x_cf,
            T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
            n_mc_path=N_MC_DESCENT, n_mc_grad=N_SAMPLES,
            sigma_start=sig_adapt, loss_stop=LOSS_STOP,
            lv_floor=lv_floor_adapt, lv_ceil=LV_CEIL,
            mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['KL-IG² (adaptive)'][idx] = absmax_collapse(r.attr).cpu()

    with open(_cache_attr, 'wb') as f: pickle.dump(all_attrs, f)
    print('Done.')

# Verify completeness
for m in METHODS:
    print(f'{m}: {len(all_attrs[m])} images')


## Attribution Maps

In [ ]:
N_VIS = 4
vis_idxs = list(range(min(N_VIS, len(dataset))))

fig, axes = plt.subplots(
    len(vis_idxs), 1 + len(METHODS),
    figsize=(2.5 * (1 + len(METHODS)), 2.5 * len(vis_idxs)),
    facecolor='white', squeeze=False,
)
for r, i in enumerate(vis_idxs):
    row_v  = dataset[i]
    img_np = np.clip(denormalize(row_v['x'][0]).permute(1,2,0).numpy(), 0, 1)
    axes[r, 0].imshow(img_np); axes[r, 0].axis('off')
    axes[r, 0].set_ylabel(imagenet_labels[row_v['target']][:18], fontsize=8)
    if r == 0: axes[r, 0].set_title('Original', fontsize=10, fontweight='bold')
    for c, m in enumerate(METHODS, start=1):
        a    = all_attrs[m][i].numpy()
        vmax = max(float(np.percentile(np.abs(a), 99)), 1e-9)
        axes[r, c].imshow(a, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[r, c].axis('off')
        if r == 0:
            axes[r, c].set_title(m, fontsize=8, fontweight='bold', color=COLORS[m])
plt.suptitle('Attribution maps — all 9 methods', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


## Sparsity (Gini)

In [ ]:
_cache_gini = CACHE_DIR / 'gini.pkl'

def gini(v):
    v = v.abs().flatten().numpy(); v = np.sort(v); n = len(v)
    return float((2*np.arange(1,n+1) - n - 1) @ v / (n * v.sum() + 1e-12))

if not FORCE_RECOMPUTE and _cache_gini.exists():
    with open(_cache_gini, 'rb') as f: gini_scores = pickle.load(f)
    print('[cache] gini loaded')
else:
    gini_scores = {m: [gini(all_attrs[m][i]) for i in range(len(dataset))] for m in METHODS}
    with open(_cache_gini, 'wb') as f: pickle.dump(gini_scores, f)

fig, ax = plt.subplots(figsize=(10, 4.5), facecolor='white')
for xi, m in enumerate(METHODS):
    v = gini_scores[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
    ax.bar(xi, mu_, color=COLORS[m], alpha=0.88, width=0.6, edgecolor='white')
    ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='#333333', capsize=4)
ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0); ax.set_axisbelow(True)
ax.set_xticks(range(len(METHODS))); ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Gini coefficient ↑', fontsize=10)
ax.set_title(f'Sparsity (Gini)  —  n={len(dataset)}', fontsize=12)
plt.tight_layout(); plt.show()

for m in METHODS:
    v = gini_scores[m]
    print(f'{m:20s}  mean={np.mean(v):.3f}  ci95=±{1.96*np.std(v)/len(v)**0.5:.3f}')


## Insertion / Deletion AUC

In [ ]:
_cache_id = CACHE_DIR / 'ins_del.pkl'

def insertion_deletion(model, x, attr_map, target, n_steps=N_INSERTION_STEPS):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    order   = attr_map.detach().view(-1).abs().argsort(descending=True)
    pps     = max(1, H * W // n_steps)
    blur    = F.avg_pool2d(x, 31, 1, 15)
    x_ins, x_del = blur.clone(), x.clone()
    ins_s, del_s = [], []
    with torch.no_grad():
        for step in range(n_steps):
            pix = order[step * pps:(step + 1) * pps]
            for ch in range(C):
                x_ins[:, ch].reshape(-1)[pix] = x[:, ch].reshape(-1)[pix]
                x_del[:, ch].reshape(-1)[pix] = blur[:, ch].reshape(-1)[pix]
            ins_s.append(model(x_ins).softmax(-1)[0, target].item())
            del_s.append(model(x_del).softmax(-1)[0, target].item())
    return float(np.trapz(ins_s) / n_steps), float(np.trapz(del_s) / n_steps)

if not FORCE_RECOMPUTE and _cache_id.exists():
    with open(_cache_id, 'rb') as f: ins_auc, del_auc = pickle.load(f)
    print('[cache] ins/del loaded')
else:
    ins_auc = defaultdict(list); del_auc = defaultdict(list)
    for row in tqdm(dataset, desc='ins/del'):
        x, tgt = row['x'], row['target']
        for m in METHODS:
            attr = all_attrs[m][row['idx']].to(DEVICE).unsqueeze(0)
            i_, d = insertion_deletion(model, x, attr, tgt)
            ins_auc[m].append(i_); del_auc[m].append(d)
    ins_auc, del_auc = dict(ins_auc), dict(del_auc)
    with open(_cache_id, 'wb') as f: pickle.dump((ins_auc, del_auc), f)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), facecolor='white')
for ax, (title, aucs) in zip(axes, [('Insertion AUC ↑', ins_auc), ('Deletion AUC ↓', del_auc)]):
    for xi, m in enumerate(METHODS):
        v = aucs[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
        ax.bar(xi, mu_, color=COLORS[m], alpha=0.88, width=0.6, edgecolor='white')
        ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='#333333', capsize=4)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0); ax.set_axisbelow(True)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
    ax.set_title(title, fontsize=12)
plt.suptitle(f'Insertion / Deletion AUC  (n={len(dataset)})', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## Sensitivity-n (baseline-matched)

In [ ]:
_cache_sn = CACHE_DIR / 'sens_n.pkl'

def sensitivity_n(model, x, attr_map, target, baseline,
                  n_subsets=50, subset_size=0.1):
    rng_sn    = np.random.default_rng(42)
    attr_flat = attr_map.cpu().detach().view(-1).numpy()
    n_pix     = attr_flat.size
    n_sel     = max(1, int(n_pix * subset_size))
    df_list, da_list = [], []
    with torch.no_grad():
        f_x = model(x).softmax(-1)[0, target].item()
        for _ in range(n_subsets):
            idx    = rng_sn.choice(n_pix, n_sel, replace=False)
            x_mask = x.clone()
            for ch in range(x.shape[1]):
                x_mask[:, ch].reshape(-1)[idx] = baseline[:, ch].reshape(-1)[idx]
            f_mask = model(x_mask).softmax(-1)[0, target].item()
            df_list.append(f_x - f_mask)
            da_list.append(float(attr_flat[idx].sum()))
    df, da = np.array(df_list), np.array(da_list)
    if df.std() < 1e-9 or da.std() < 1e-9: return 0.0
    return float(np.corrcoef(df, da)[0, 1])

if not FORCE_RECOMPUTE and _cache_sn.exists():
    with open(_cache_sn, 'rb') as f: sens_n = pickle.load(f)
    print('[cache] sens_n loaded')
else:
    sens_n  = defaultdict(list)
    zeros   = None
    for row in tqdm(dataset, desc='sensitivity-n'):
        x, tgt = row['x'], row['target']
        if zeros is None: zeros = torch.zeros_like(x)
        x_cf_i = pick_cf_image(row['idx'])
        if x_cf_i.dim() == 3: x_cf_i = x_cf_i.unsqueeze(0)
        baselines = {m: zeros for m in METHODS}
        baselines['KL-IG²'] = x_cf_i.to(DEVICE)
        baselines['KL-IG² (adaptive)'] = x_cf_i.to(DEVICE)
        for m in METHODS:
            if row['idx'] not in all_attrs.get(m, {}): continue
            attr = all_attrs[m][row['idx']].to(DEVICE).unsqueeze(0)
            sens_n[m].append(sensitivity_n(model, x, attr, tgt, baselines[m]))
    sens_n = dict(sens_n)
    with open(_cache_sn, 'wb') as f: pickle.dump(sens_n, f)

fig, ax = plt.subplots(figsize=(10, 4.5), facecolor='white')
for xi, m in enumerate(METHODS):
    v = sens_n[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
    ax.bar(xi, mu_, color=COLORS[m], alpha=0.88, width=0.6, edgecolor='white')
    ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='#333333', capsize=4)
ax.axhline(0, color='black', lw=0.7)
ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0); ax.set_axisbelow(True)
ax.set_xticks(range(len(METHODS))); ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Pearson r (signed) ↑', fontsize=10)
ax.set_title(f'Sensitivity-n (baseline-matched,  n={len(dataset)})', fontsize=12)
plt.tight_layout(); plt.show()


## OFR (Object Focus Ratio)

In [ ]:
import cv2

_cache_ofr = CACHE_DIR / 'ofr.pkl'

def estimate_object_mask(x, attr_map):
    """GrabCut-based object mask seeded by KLIG-Adaptive attribution percentiles."""
    H, W = attr_map.shape
    a    = np.abs(attr_map.detach().cpu().numpy())
    seed = (a >= np.percentile(a, 80)).astype(np.uint8)
    img_rgb = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)
    img_bgr = (img_rgb * 255).clip(0, 255).astype(np.uint8)[:, :, ::-1].copy()
    gc_mask = np.where(seed, cv2.GC_PR_FGD, cv2.GC_PR_BGD).astype(np.uint8)
    gc_mask[(a >= np.percentile(a, 95))] = cv2.GC_FGD
    border  = max(H, W) // 10
    edge    = np.zeros((H, W), dtype=np.uint8)
    edge[:border, :] = 1; edge[-border:, :] = 1
    edge[:, :border] = 1; edge[:, -border:] = 1
    gc_mask[(edge == 1) & (a < np.percentile(a, 10))] = cv2.GC_BGD
    try:
        bgd = np.zeros((1, 65), np.float64)
        fgd = np.zeros((1, 65), np.float64)
        cv2.grabCut(img_bgr, gc_mask, None, bgd, fgd, 5, cv2.GC_INIT_WITH_MASK)
        return np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)
    except Exception:
        return seed

def object_focus_ratio(attr_map, obj_mask):
    """Fraction of |attr| mass inside the object mask."""
    a = np.abs(attr_map.detach().cpu().numpy())
    total = a.sum()
    return float(a[obj_mask == 1].sum() / total) if total > 1e-12 else 0.0

if not FORCE_RECOMPUTE and _cache_ofr.exists():
    with open(_cache_ofr, 'rb') as f: ofr_scores = pickle.load(f)
    print('[cache] OFR loaded')
else:
    ofr_scores = {m: [] for m in METHODS}
    for row in tqdm(dataset, desc='OFR'):
        x, tgt = row['x'], row['target']
        # Reference mask: use KLIG-Adaptive attribution as GrabCut seed
        ref_attr = all_attrs['KLIG-Adaptive'][row['idx']].to(DEVICE)
        obj_mask = estimate_object_mask(x, ref_attr)
        for m in METHODS:
            attr = all_attrs[m][row['idx']]
            ofr_scores[m].append(object_focus_ratio(attr, obj_mask))
    with open(_cache_ofr, 'wb') as f: pickle.dump(ofr_scores, f)
    print('OFR computed and cached.')

fig, ax = plt.subplots(figsize=(10, 4.5), facecolor='white')
for xi, m in enumerate(METHODS):
    v   = ofr_scores[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
    ax.bar(xi, mu_, color=COLORS[m], alpha=0.88, width=0.6, edgecolor='white')
    ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='#333333', capsize=4)
ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0); ax.set_axisbelow(True)
ax.set_xticks(range(len(METHODS))); ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('OFR ↑  (fraction of |attr| inside object)', fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_title(f'Object Focus Ratio  (n={len(dataset)})', fontsize=12)
plt.tight_layout(); plt.show()

for m in METHODS:
    v = ofr_scores[m]
    print(f'{m:20s}  mean={np.mean(v):.3f}  ci95=±{1.96*np.std(v)/len(v)**0.5:.3f}')


## Class Sensitivity — WordNet Scatter

In [ ]:
# ── Class Sensitivity: Step 1 — filter multi-class images ────────────────────
_cache_multi = CACHE_DIR / 'multiprob.pkl'

def label_to_synset(label):
    phrase = label.lower().split(',')[0].strip()
    cands  = []
    for candidate in [phrase.replace(' ', '_'), phrase]:
        cands.extend(wn.synsets(candidate, pos=wn.NOUN))
    if not cands:
        for w in sorted(phrase.split(), key=len, reverse=True):
            if len(w) > 3:
                cands.extend(wn.synsets(w, pos=wn.NOUN))
                if cands: break
    if not cands: return None
    return max(cands, key=lambda s: s.min_depth())

cls_synsets = {i: label_to_synset(imagenet_labels[i]) for i in range(1000)}

def cosine_dist_cs(a_i, a_j):
    ai = np.clip(a_i.astype(np.float64).ravel(), 0, None)
    aj = np.clip(a_j.astype(np.float64).ravel(), 0, None)
    denom = np.linalg.norm(ai) * np.linalg.norm(aj)
    if denom < 1e-12: return 1.0
    return float(1.0 - (ai @ aj) / denom)

if not FORCE_RECOMPUTE and _cache_multi.exists():
    with open(_cache_multi, 'rb') as f: multi_imgs = pickle.load(f)
    print(f'[cache] {len(multi_imgs)} multi-class images')
else:
    multi_imgs = []
    for row in dataset:
        x = row['x']
        with torch.no_grad():
            probs = model(x).softmax(-1)[0].cpu()
        high = (probs > CS_PROB_THRESH).nonzero(as_tuple=True)[0].tolist()
        if len(high) < 2: continue
        high = sorted(high, key=lambda c: probs[c].item(), reverse=True)
        multi_imgs.append({
            'idx':        row['idx'],
            'x':          x,
            'high_cls':   high,
            'high_probs': [probs[c].item() for c in high],
        })
    with open(_cache_multi, 'wb') as f: pickle.dump(multi_imgs, f)
    print(f'Found {len(multi_imgs)}/{len(dataset)} images with ≥2 classes > {CS_PROB_THRESH}')

n_pairs_total = sum(len(list(itertools.combinations(d['high_cls'], 2))) for d in multi_imgs)
print(f'Total class pairs: {n_pairs_total}')


In [ ]:
# ── Class Sensitivity: Step 2 — scatter data computation ─────────────────────
_cache_scatter   = CACHE_DIR / 'cs_scatter.pkl'
_cache_scatter_n = CACHE_DIR / 'cs_scatter_n.pkl'

def _attr_for_class_all_methods(x1, tgt_cls, x_cf_img, sig_adapt):
    """Return dict {method: attr_flat_numpy} for a given target class."""
    result = {}
    # KLIG-Adaptive
    r = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=sig_adapt, path=LinearPath(), device=DEVICE).attribute(x1, target=tgt_cls)
    result['KLIG-Adaptive'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # KLIG-Linear
    r = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=SIGMA_FINAL, path=LinearPath(), device=DEVICE).attribute(x1, target=tgt_cls)
    result['KLIG-Linear'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # DDPath-cos
    r = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=SIGMA_FINAL, path=DDiffusionPath(schedule='cosine'), device=DEVICE).attribute(x1, target=tgt_cls)
    result['DDPath-cos'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # DDPath-lin
    r = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=SIGMA_FINAL, path=DDiffusionPath(schedule='linear'), device=DEVICE).attribute(x1, target=tgt_cls)
    result['DDPath-lin'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # DDPath-quad
    r = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=SIGMA_FINAL, path=DDiffusionPath(schedule='quadratic'), device=DEVICE).attribute(x1, target=tgt_cls)
    result['DDPath-quad'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # Greedy-μ
    r = GreedyMuAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=sig_adapt, device=DEVICE).attribute(x1, target=tgt_cls)
    result['Greedy-μ'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # Greedy-Jt
    r = GreedyJointAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=sig_adapt, device=DEVICE).attribute(x1, target=tgt_cls)
    result['Greedy-Jt'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # Greedy-Srt
    sp = SortedDimPath.from_model_and_input(model, x1, target=tgt_cls,
        n_samples=SORTED_DIM_SAMPLES, gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI)
    r = KLIntegratedGradients(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
        sigma_final=sig_adapt, path=sp, device=DEVICE).attribute(x1, target=tgt_cls)
    result['Greedy-Srt'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # KL-IG²: KLIGSquared (forward integration, model-derived baseline)
    r = KLIGSquared(model, phi, x_cf_img,
        T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
        n_mc_path=N_MC_DESCENT, n_mc_grad=N_SAMPLES,
        sigma_start=SIGMA_FINAL, loss_stop=LOSS_STOP,
        lv_floor=LV_FLOOR, lv_ceil=LV_CEIL,
        mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE
    ).attribute(x1, target=tgt_cls)
    result['KL-IG²'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    # KL-IG² (adaptive): KLIGSquared with adaptive sigma_start
    lv_floor_adapt = 2 * math.log(sig_adapt)
    r = KLIGSquared(model, phi, x_cf_img,
        T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
        n_mc_path=N_MC_DESCENT, n_mc_grad=N_SAMPLES,
        sigma_start=sig_adapt, loss_stop=LOSS_STOP,
        lv_floor=lv_floor_adapt, lv_ceil=LV_CEIL,
        mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE
    ).attribute(x1, target=tgt_cls)
    result['KL-IG² (adaptive)'] = absmax_collapse(r.attr).cpu().numpy().ravel()
    return result

if not FORCE_RECOMPUTE and _cache_scatter.exists():
    with open(_cache_scatter, 'rb') as f: scatter_all = pickle.load(f)
    with open(_cache_scatter_n, 'rb') as f: done_n = pickle.load(f)
    print(f'[resume] {sum(len(v) for v in scatter_all.values())} points, '
          f'{done_n}/{len(multi_imgs)} images done')
else:
    scatter_all = {m: [] for m in METHODS}
    done_n      = 0

for d in tqdm(multi_imgs[done_n:], desc='class-sens scatter'):
    x1       = d['x'].squeeze(0).to(DEVICE)
    high_cls = d['high_cls']
    idx      = d['idx']
    sig_adapt = sigma_per_idx[idx]
    x_cf_img = pick_cf_image(idx)
    if x_cf_img.dim() == 4: x_cf_img = x_cf_img.squeeze(0)
    x_cf_img = x_cf_img.to(DEVICE)

    # Pre-compute all (class → attr) for all methods for this image
    attr_cache = {}
    for cls in high_cls:
        attr_cache[cls] = _attr_for_class_all_methods(x1, cls, x_cf_img, sig_adapt)

    for ci, cj in itertools.combinations(high_cls, 2):
        s_i = cls_synsets.get(ci)
        s_j = cls_synsets.get(cj)
        if s_i is None or s_j is None: continue
        wn_dist = s_i.shortest_path_distance(s_j)
        if wn_dist is None: continue
        for m in METHODS:
            cd = cosine_dist_cs(attr_cache[ci][m], attr_cache[cj][m])
            scatter_all[m].append((int(wn_dist), cd))

    done_n += 1
    with open(_cache_scatter, 'wb') as f: pickle.dump(scatter_all, f)
    with open(_cache_scatter_n, 'wb') as f: pickle.dump(done_n, f)

print(f'Scatter data ready — {len(scatter_all[METHODS[0]])} points per method')


In [ ]:
# ── Class Sensitivity: Step 3 — 3×3 scatter plot ─────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(18, 15), facecolor='white', squeeze=False)
rng_jitter = np.random.default_rng(0)

for ax_flat_idx, m in enumerate(METHODS):
    ax = axes[ax_flat_idx // 3][ax_flat_idx % 3]
    pts = scatter_all[m]
    if not pts:
        ax.set_title(f'{m}\nn/a', fontsize=10); continue
    dists = np.array([p[0] for p in pts])
    divs  = np.array([p[1] for p in pts])
    jitter = rng_jitter.uniform(-0.15, 0.15, size=len(dists))
    ax.scatter(dists + jitter, divs, alpha=0.35, s=18,
               c=dists, cmap='plasma', edgecolors='none')
    for d_val in sorted(set(dists.tolist())):
        vals = divs[dists == d_val]
        if len(vals) >= 2:
            ax.errorbar(d_val, vals.mean(), yerr=vals.std(),
                        fmt='o', color='black', ms=5, lw=1.4, zorder=5, capsize=3)
        else:
            ax.scatter([d_val], [vals.mean()], color='black', s=40, zorder=5)
    if len(set(dists.tolist())) > 1:
        z  = np.polyfit(dists, divs, 1)
        xs = np.linspace(dists.min(), dists.max(), 100)
        ax.plot(xs, np.poly1d(z)(xs), color=COLORS[m], lw=2, ls='--', alpha=0.8)
    rho_val, pval = spearmanr(dists, divs)
    ax.set_title(f'{m}\nρ = {rho_val:.3f}  (p = {pval:.3f})  n={len(pts)}',
                 fontsize=10, fontweight='bold', color=COLORS[m])
    ax.set_xlabel('WordNet shortest-path distance', fontsize=9)
    ax.set_ylabel('Cosine distance (attrs)', fontsize=9)
    ax.set_xlim(dists.min() - 0.5, dists.max() + 0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0); ax.set_axisbelow(True)

plt.suptitle(
    'Class Sensitivity: Attribution Cosine Distance vs. WordNet Semantic Distance\n'
    'Positive ρ → method responds to class identity | higher = more class-discriminative',
    fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()

print(f"{'Method':<20}  {'rho':>8}  {'p':>10}  {'n':>6}")
print('-' * 50)
for m in METHODS:
    pts = scatter_all[m]
    if not pts: print(f'{m:<20}  n/a'); continue
    dists = np.array([p[0] for p in pts]); divs = np.array([p[1] for p in pts])
    rho_val, pval = spearmanr(dists, divs)
    print(f'{m:<20}  {rho_val:>8.3f}  {pval:>10.4f}  {len(pts):>6}')


## Path Attribution Change Curves

In [ ]:
# ── Path attribution change curves ────────────────────────────────────────────
_cache_pc = CACHE_DIR / 'path_curves.pkl'

def _path_step_signal(path, x1, target, sigma, n_steps, n_samples):
    """Walk path, return per-step |g_mu*dmu + g_lv*dlv| array."""
    model.eval()
    saved = [p.requires_grad for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(False)
    lv_final = torch.full_like(x1, 2.0 * math.log(sigma))
    ts   = path.steps(n_steps).tolist()
    dt   = 1.0 / n_steps
    sigs = []
    try:
        for t in ts:
            mu_t, lv_t   = path.at(t, x1, lv_final)
            dmu_t, dlv_t = path.derivatives(t, x1, lv_final)
            mu_g = mu_t.detach().requires_grad_(True)
            lv_g = lv_t.detach().requires_grad_(True)
            eps  = torch.randn(n_samples, *x1.shape, device=x1.device)
            xs   = mu_g.unsqueeze(0) + (0.5*lv_g).exp().unsqueeze(0) * eps
            model(xs)[:, target].mean().backward()
            sig = float((mu_g.grad * dmu_t * dt + lv_g.grad * dlv_t * dt).abs().sum().item())
            sigs.append(sig)
    finally:
        for p, s in zip(model.parameters(), saved): p.requires_grad_(s)
    return np.array(sigs)


def get_curve_signal(m, row):
    """Compute path change signal for a single method + image."""
    x, tgt = row['x'], row['target']
    idx = row['idx']
    x1  = x.squeeze(0).to(DEVICE)
    sig_adapt = sigma_per_idx[idx]

    if m == 'KLIG-Adaptive':
        return _path_step_signal(LinearPath(), x1, tgt, sig_adapt, N_STEPS, N_SAMPLES)
    elif m == 'KLIG-Linear':
        return _path_step_signal(LinearPath(), x1, tgt, SIGMA_FINAL, N_STEPS, N_SAMPLES)
    elif m == 'DDPath-cos':
        return _path_step_signal(DDiffusionPath('cosine'), x1, tgt, SIGMA_FINAL, N_STEPS, N_SAMPLES)
    elif m == 'DDPath-lin':
        return _path_step_signal(DDiffusionPath('linear'), x1, tgt, SIGMA_FINAL, N_STEPS, N_SAMPLES)
    elif m == 'DDPath-quad':
        return _path_step_signal(DDiffusionPath('quadratic'), x1, tgt, SIGMA_FINAL, N_STEPS, N_SAMPLES)
    elif m == 'Greedy-μ':
        r = GreedyMuAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sig_adapt, device=DEVICE).attribute(x1, target=tgt)
        return np.array(r.step_grad_signal)
    elif m == 'Greedy-Jt':
        r = GreedyJointAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sig_adapt, device=DEVICE).attribute(x1, target=tgt)
        return np.array(r.step_grad_signal)
    elif m == 'Greedy-Srt':
        sp = SortedDimPath.from_model_and_input(model, x1, target=tgt,
            n_samples=SORTED_DIM_SAMPLES, gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI)
        return _path_step_signal(sp, x1, tgt, sig_adapt, N_STEPS, N_SAMPLES)
    elif m == 'KL-IG²':
        x_cf = pick_cf_image(idx)
        if x_cf.dim() == 4: x_cf = x_cf.squeeze(0)
        x_cf = x_cf.to(DEVICE)
        r = KLIGSquared(model, phi, x_cf,
            T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
            n_mc_path=N_MC_DESCENT, n_mc_grad=N_SAMPLES,
            sigma_start=SIGMA_FINAL, loss_stop=LOSS_STOP,
            lv_floor=LV_FLOOR, lv_ceil=LV_CEIL,
            mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE
        ).attribute(x1, target=tgt)
        return np.array(r.step_grad_signal)
    elif m == 'KL-IG² (adaptive)':
        x_cf = pick_cf_image(idx)
        if x_cf.dim() == 4: x_cf = x_cf.squeeze(0)
        x_cf = x_cf.to(DEVICE)
        lv_floor_adapt = 2 * math.log(sig_adapt)
        r = KLIGSquared(model, phi, x_cf,
            T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
            n_mc_path=N_MC_DESCENT, n_mc_grad=N_SAMPLES,
            sigma_start=sig_adapt, loss_stop=LOSS_STOP,
            lv_floor=lv_floor_adapt, lv_ceil=LV_CEIL,
            mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE
        ).attribute(x1, target=tgt)
        return np.array(r.step_grad_signal)
    raise ValueError(f'Unknown method: {m}')

if not FORCE_RECOMPUTE and _cache_pc.exists():
    with open(_cache_pc, 'rb') as f: curve_signals = pickle.load(f)
    print(f'[cache] path curves loaded')
else:
    curve_signals = {m: [] for m in METHODS}
    for row in tqdm(dataset[:N_CURVE_IMGS], desc='path-curves'):
        for m in METHODS:
            try:
                curve_signals[m].append(get_curve_signal(m, row))
            except Exception as e:
                print(f'  [{m}] img {row["idx"]}: {e}')
    with open(_cache_pc, 'wb') as f: pickle.dump(curve_signals, f)
    print(f'Path curves computed for {N_CURVE_IMGS} images.')


In [ ]:
# ── Path change curves — absolute (log-scale) + normalized ──────────────────
_n   = N_STEPS
xs   = np.linspace(0, 1, _n)
uni  = 1.0 / _n

fig, axes = plt.subplots(1, 2, figsize=(15, 5), facecolor='white')

for m in METHODS:
    curves = curve_signals.get(m, [])
    if not curves: continue
    # normalize per image then average
    normed = []
    for c in curves:
        c = np.asarray(c, dtype=float)
        if len(c) < _n: c = np.pad(c, (0, _n - len(c)))
        else: c = c[:_n]
        total = c.sum() + 1e-12
        normed.append(c / total)
    mat  = np.stack(normed)
    mean = mat.mean(0)
    sem  = mat.std(0) / math.sqrt(len(mat))

    # Panel 1: normalized per-step signal
    axes[0].plot(xs, mean, color=COLORS[m], lw=2, label=m)
    axes[0].fill_between(xs, mean - sem, mean + sem, color=COLORS[m], alpha=0.12)

    # Panel 2: log-scale absolute (un-normalized mean)
    raw = np.stack([np.asarray(c, dtype=float)[:_n] if len(c) >= _n
                    else np.pad(np.asarray(c, dtype=float), (0, _n-len(c)))
                    for c in curves]).mean(0)
    axes[1].plot(xs, np.maximum(raw, 1e-12), color=COLORS[m], lw=2, label=m)

axes[0].axhline(uni, color='gray', lw=1, ls='--', label='uniform')
axes[0].set_xlabel('Path position s', fontsize=11)
axes[0].set_ylabel('Normalised step signal', fontsize=11)
axes[0].set_title(f'Path attribution change (normalised)  n={N_CURVE_IMGS}', fontsize=12)
axes[0].legend(fontsize=8, ncol=2); axes[0].grid(alpha=0.25); axes[0].set_xlim(0, 1)

axes[1].set_yscale('log')
axes[1].set_xlabel('Path position s', fontsize=11)
axes[1].set_ylabel('|step signal|  (log scale)', fontsize=11)
axes[1].set_title('Absolute step signal (log scale)', fontsize=12)
axes[1].legend(fontsize=8, ncol=2); axes[1].grid(alpha=0.25); axes[1].set_xlim(0, 1)

plt.suptitle('Path Attribution Change Curves — all 9 methods', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## Summary Table

In [ ]:
import pandas as pd

ci95 = lambda v: 1.96 * np.std(v) / (len(v) ** 0.5)
rows_sum = []
for m in METHODS:
    r = {'Method': m}
    if gini_scores.get(m):
        v = gini_scores[m]; r['Gini ↑'] = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if ins_auc.get(m):
        v = ins_auc[m]; r['Ins AUC ↑'] = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if del_auc.get(m):
        v = del_auc[m]; r['Del AUC ↓'] = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if sens_n.get(m):
        v = sens_n[m]; r['Sens-n ↑'] = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if ofr_scores.get(m):
        v = ofr_scores[m]; r['OFR ↑'] = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if scatter_all.get(m) and scatter_all[m]:
        pts   = scatter_all[m]
        dists = np.array([p[0] for p in pts]); divs = np.array([p[1] for p in pts])
        rho_, _ = spearmanr(dists, divs)
        r['CS ρ ↑'] = f'{rho_:.3f}'
    rows_sum.append(r)

df = pd.DataFrame(rows_sum).set_index('Method')
print(df.to_string())
df
